## CSC 466: Knowledge Discovery in Data
## Cal Poly, San Luis Obispo
### Spring 2026

# Lab 2


### Due Thursday, April 30, 3:00pm




## Before You Do Anything Else!!!

This is the Instructor's version of this notebook.  **You need to save a copy of this notebook** to your personal Google Drive. To do this, go to File > Save a copy in Drive.


**Name:** Winnie Trinh

**Cal Poyl Email:** witrinh@calpoly.edu

**Name:** Nathan Madlansacay

**Cal Poyl Email:** nmadlans@calpoly.edu




In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
### import specifc sklearn classes below

from sklearn import datasets
from collections import Counter
from math import log2
from pandas.api.types import is_numeric_dtype
import json

## Instructions

Develop and test all functionality needed to complete **Lab 2** in the code and Markdown cells below as specfied in the lab assignment. Use as many code/Markdown cells as you need.


In [ ]:
%%writefile c45.py

import json
import pandas as pd
from math import log2
from collections import Counter
from pandas.api.types import is_numeric_dtype
class c45:
  def __init__(self, metric="InfoGain", threshold=0.0, featureType=None):
    """
    Parameters:
    metric: str - "InfoGain" or "Ratio"

    threshold: float
    create node if metric > threshold

    featureType: dictionary - "categorical" or "numeric"
    """
    # check if correct desired metric is allowed
    allowedMetrics = ["InfoGain", "Ratio"]
    if metric not in allowedMetrics:
        raise ValueError(f"metric must be one of {allowedMetrics}")

    # check if threshold is a valid number
    if not isinstance(threshold, (int, float)) or threshold < 0.0:
        raise ValueError(f"threshold must be a nonnegative number")

    self.metric = metric
    self.threshold = threshold
    self.featureType = featureType
    self.defaultClass = None

    #final decision tree after calling fit()
    self.tree = None

  # determine whether each feature is numeric or categorical
  def determineFeatures(self, X):
    features = {}

    # loop through each column name in DataFrame
    for col in X.columns:
      if is_numeric_dtype(X[col]):
        features[col] = "numeric"
      else:
        features[col] = "categorical"

    return features

  # trains and builds the C45 decision tree
  def fit(self, Xtrain, ytrain, data=None, featureType=None):
    """
    Builds C4.5 decision tree using the training data

    Parameters:
    Xtrain: DataFrame of independent variables
    ytrain: Series/list of class labels
    data: optional name of the dataset
    """
    # convert into DataFrame and reset row numbers to 0, 1, ...
    Xtrain = pd.DataFrame(Xtrain).reset_index(drop=True)
    # convert to Series and reset row numbers to match ^
    ytrain = pd.Series(ytrain).reset_index(drop=True)

    self.defaultClass = self.majorityClass(ytrain)

    # use featureType dict or determine them
    if featureType is not None:
        self.featureType = featureType
    elif self.featureType is None:
        self.featureType = self.determineFeatures(Xtrain)

    # build tree using columns as possible splitting attributes
    root = self.buildTree(Xtrain, ytrain, list(Xtrain.columns))

    # store tree in json format
    self.tree = {
        "dataset": data,
        "node": root
    }

    return self.tree

  # calculate entropy, how unorganized class labels are
  def entropy(self, y):
    #case 1: if there are no labels
    if len(y) == 0:
      return 0.0

    # count number of times label appears
    total = len(y)
    counts = Counter(y)

    entropy = 0.0

    # calculate proportion of class
    for count in counts.values():
      proportion = count / total
      entropy += -proportion * log2(proportion)

    return entropy

  def infoGain(self, X, y, attribute, threshold = None):
    """
    Calculates information gain of splitting on an attribute.

    X: DataFrame of features
    y: labels
    attribute: str - column name to split on
    threshold: float or None
      None -> categorical split
      float -> numeric split at value
    """
    # calculate entrop before attribute split
    pEntropy = self.entropy(y)

    # total number of rows
    total = len(y)

    # no infogain if there are no labels
    if total == 0:
      return 0.0

    # entropy after attribute split
    weightedEntropy = 0.0

    # treat as categorical
    if threshold is None:
      # loop through each value for attribute
      for value in X[attribute].unique():
        subset = y[X[attribute] == value]
        weightedEntropy += (len(subset) / total) * self.entropy(subset)

    # treat as numeric attribute
    else:
      # create D- and D+ subsets
      leftSubset = y[X[attribute] <= threshold]
      rightSubset = y[X[attribute] > threshold]

      weightedEntropy += (len(leftSubset) / total) * self.entropy(leftSubset)
      weightedEntropy += (len(rightSubset) / total) * self.entropy(rightSubset)

    return pEntropy - weightedEntropy

  # calculate split value which is denominator for gainRatio
  def splitValue(self, X, attribute, threshold=None):
    # total number of rows
    total = len(X)

    # return 0.0 if no rows
    if total == 0:
      return 0.0

    splitVal = 0.0

    # treat as categorical split
    if threshold is None:
      # loop through each value for attribute
      for value in X[attribute].unique():
        proportion = len(X[X[attribute] == value]) / total

        if proportion > 0:
          splitVal += -proportion * log2(proportion)

    # treat as numeric split
    else:
      # create D- and D+ subsets
      leftSubset = X[X[attribute] <= threshold]
      rightSubset = X[X[attribute] > threshold]

      # loop through subsets and calculate the proportions
      for subset in [leftSubset, rightSubset]:
        proportion = len(subset) / total

        if proportion > 0:
          splitVal += -proportion * log2(proportion)

    return splitVal

  # calculate gain ratio, other option to infogain
  def gainRatio(self, X, y, attribute, threshold=None):
    """
    Calculates gain ratio of splitting on an attribute
    """

    # calculate information gain first
    gain = self.infoGain(X, y, attribute, threshold)
    # then calculate split value
    splitVal = self.splitValue(X, attribute, threshold)

    if splitVal == 0.0:
      return 0.0

    # return ratio
    return gain / splitVal

  def majorityClass(self, y):
    """
    Returns the majority class
    """

    # return None if no labels
    if len(y) == 0:
      return None

    # return most command class label
    return Counter(y).most_common(1)[0][0]

  def possibleThresholds(self, X, attribute):
    """
    Returns a list of possible thresholds for a numeric attribute
    """

    values = sorted(X[attribute].dropna().unique())
    thresholds = []

    # loop through each pair of values
    for i in range(len(values) - 1):
      # add midpoints between values
      thresholds.append((values[i] + values[i + 1]) / 2)

    return thresholds

  def findBestSplit(self, X, y, attributes):
    """
    Finds the best attribute to split on
    """
    best = None
    bestGain = 0.0
    bestThreshold = None

    # loop through each attribute
    for attribute in attributes:
      attributeType = self.featureType[attribute]

      # treat as numeric attribute
      if attributeType == "numeric":
        # find possible thresholds
        thresholds = self.possibleThresholds(X, attribute)
        for t in thresholds:
          if self.metric == "InfoGain":
            gain = self.infoGain(X, y, attribute, t)
          else:
            gain = self.gainRatio(X, y, attribute, t)

          # save information if this is the best gain score so far
          if gain > bestGain:
            best = attribute
            bestGain = gain
            bestThreshold = t

      # treat as categorical attributes, no thresholds
      else:
        if self.metric == "InfoGain":
          gain = self.infoGain(X, y, attribute)
        else:
          gain = self.gainRatio(X, y, attribute)

        # save information if this is the best gain score so far
        if gain > bestGain:
          best = attribute
          bestGain = gain
          bestThreshold = None

    return best, bestThreshold, bestGain

  # tree building function and helper functions

  # create leaf node, this is the final prediction
  def leaf(self, y):
    """
    Returns a leaf node with the majority class label
    """

    # get most common class label
    mostCommon = self.majorityClass(y)


    # calculate probability
    if len(y) == 0:
      prob = 0.0

    else:
      prob = Counter(y)[mostCommon] / len(y)

    return {
        "leaf": {
            "decision": mostCommon,
            "p": prob
        }
    }

  # create edge between a parent and child
  def edge(self, val, op, child):
    """
    Creates an edge node

    val: category/numeric threshold
    op: attribute name
    child: child node/leaf
    """

    # edge has a value
    edge = {
        "value": val
    }
    if op is not None:
      edge["op"] = op

    # store child as leaf or decision node
    if "leaf" in child:
      edge["leaf"] = child["leaf"]
    else:
      edge["node"] = child

    return {
        "edge": edge
    }

  # build decision tree
  def buildTree(self, X, y, attributes, parent=None):
    """
    Build decision tree

    X: DataFrame of features
    y: labels
    attributes: list of attributes for splitting
    """

    # no data left
    if len(y) == 0:
      return self.leaf(y)

    # no attributes left
    if len(attributes) == 0:
      return self.leaf(y)

    # all labels are the same
    if len(set(y)) == 1:
      return self.leaf(y)

    best, bestThreshold, bestGain = self.findBestSplit(X, y, attributes)

    # no attribute gives good split
    if best is None or bestGain <= self.threshold:
      return self.leaf(y)

    node = {
        "var": best,
        "type": self.featureType[best],
        "mostCommon": self.majorityClass(y),
        "edges": []
    }

    # numberic split
    if self.featureType[best] == "numeric":
      # create left and right subtrees
      leftChild = self.buildTree(X[X[best] <= bestThreshold].reset_index(drop=True),
                                 y[X[best] <= bestThreshold].reset_index(drop=True),
                                 attributes)
      rightChild = self.buildTree(X[X[best] > bestThreshold].reset_index(drop=True),
                                  y[X[best] > bestThreshold].reset_index(drop=True),
                                  attributes)

      # add edges connecting node to left and right subtrees
      node["edges"].append(self.edge(bestThreshold, "<=", leftChild))
      node["edges"].append(self.edge(bestThreshold, ">", rightChild))

    # categorical split
    else:
      # remove category after splitting
      newAttributes = attributes.copy()
      newAttributes.remove(best)

      # create a branch per value in the category
      for val in X[best].unique():
        child = self.buildTree(X[X[best] == val].reset_index(drop=True),
                               y[X[best] == val].reset_index(drop=True),
                               newAttributes)
        # add edge to connect new category branch
        node["edges"].append(self.edge(val, None, child))

    return node

  # prediction functions
  def predict(self, Xtest):
    """
    Predicts class labels for new data

    Xtest: DataFrame of test data
    """

    if self.tree is None:
      raise ValueError("Decision tree not trained")

    # convert test data into DataFrame
    Xtest = pd.DataFrame(Xtest).reset_index(drop=True)

    preds = []

    # for each row, predict what final decision would be
    for i, row in Xtest.iterrows():
      preds.append(self.predictRow(row, self.tree["node"]))

    return preds

  # helper function to predict, predicts for a singular row
  def predictRow(self, row, node):
    """
    Predicts class label for a single row

    row: Series of row data
    node: current node in tree
    """

    # return decision if node is already a leaf
    if "leaf" in node:
      return node["leaf"]["decision"]

    attr = node["var"]
    attrType = node["type"]
    attrVal = row[attr]

    # numeric split
    if attrType == "numeric":
      # go through every edge that connects to this node
      for edge in node["edges"]:
        e = edge["edge"]
        threshold = e["value"]
        op = e["op"]

        # <= branch
        if op == "<=" and attrVal <= threshold:
          if "leaf" in e:
            return e["leaf"]["decision"]
          else:
            return self.predictRow(row, e["node"])

        # > branch
        if op == ">" and attrVal > threshold:
          if "leaf" in e:
            return e["leaf"]["decision"]
          else:
            return self.predictRow(row, e["node"])
    # categorical split
    else:
      for edge in node["edges"]:
        e = edge["edge"]

        if attrVal == e["value"]:
          if "leaf" in e:
            return e["leaf"]["decision"]
          else:
            return self.predictRow(row, e["node"])

    # return most common label if the other return statements don't execute
    return node.get("mostCommon", self.defaultClass)

  # save and read json
  def read_tree(self, file):
    """
    Reads in a decision tree from a json file
    """
    with open(file, "r") as f:
      self.tree = json.load(f)

    return self.tree

  def save_tree(self, file):
    """
    Saves a decision tree to a json file
    """
    if self.tree is None:
      raise ValueError("Decision tree not trained")

    with open(file, "w") as f:
      json.dump(self.tree, f, indent=4)


Writing c45.py


In [ ]:
%%writefile utils.py

import pandas as pd
def parseCSV(filepath):
    """
    Parses the custom 3-line header CSV format.
    Returns X (DataFrame), y (Series), featureType (dict), classVar (str)
    """
    # read first 3 header lines
    with open(filepath, "r") as f:
      lines = f.readlines()

    # get column names from first line
    colNames = [c.strip() for c in lines[0].strip().split(',')]

    # get domain codes from second line
    domainCodes = list(map(int, lines[1].strip().split(",")))

    # get class variable name from the 3rd line
    classVar = lines[2].strip()

    data = pd.read_csv(filepath, skiprows=3, header=None, names=colNames)

    dropCols = []

    for i, col in enumerate(domainCodes):
      if col == -1:
        dropCols.append(colNames[i])

    data = data.drop(columns=dropCols)

    featureType = {}

    for i, col in enumerate(colNames):
      if domainCodes[i] == -1:
        continue

      if col == classVar:
        continue

      if domainCodes[i] == 0:
        featureType[col] = "numeric"
        data[col] = pd.to_numeric(data[col])
      else:
        featureType[col] = "categorical"

    # split into X and y
    # y will be None if classVar isn't in data (e.g. pure prediction set with no labels)
    if classVar in data.columns:
        X = data.drop(columns=[classVar])
        y = data[classVar]
    else:
        X = data
        y = None

    return X, y, featureType, classVar

Writing utils.py


In [ ]:
%%writefile predict.py
import sys
from collections import defaultdict

from c45 import c45
from utils import parseCSV
def printConfusionMatrix(matrix, classes):
    """
    Prints a formatted confusion matrix.

    matrix: dict of dict  {actual: {predicted: count}}
    classes: sorted list of unique class labels
    """
    # column width based on longest class name
    colWidth = max(len(str(c)) for c in classes) + 2

    # header row
    header = "Actual \\ Predicted".ljust(colWidth)
    for c in classes:
        header += str(c).rjust(colWidth)
    print(header)
    print("-" * len(header))

    # one row per actual class
    for actual in classes:
        row = str(actual).ljust(colWidth)
        for predicted in classes:
            count = matrix[actual][predicted]
            row += str(count).rjust(colWidth)
        print(row)

def main():
    if len(sys.argv) < 3:
        print("Usage: python predict.py <CSVFile> <JSONFile> [eval]")
        sys.exit(1)

    csvFile  = sys.argv[1]
    jsonFile = sys.argv[2]
    evalMode = len(sys.argv) > 3 and sys.argv[3].lower() == "eval"

    #  load the decision tree from JSON
    model = c45()
    model.read_tree(jsonFile)

    #  parse the CSV file
    X, y, featureType, classVar = parseCSV(csvFile)

    #  make predictions
    predictions = model.predict(X)

    #  basic output: one prediction per line
    if not evalMode:
        for pred in predictions:
            print(pred)
        return

    #  eval mode: need ground truth
    if y is None:
        print("Error: eval mode requires ground truth labels in the CSV file.")
        sys.exit(1)

    groundTruth = list(y)

    # print predictions alongside ground truth
    print(f"{'Row':<6} {'Actual':<20} {'Predicted':<20} {'Correct'}")
    print("-" * 60)
    for i, (actual, predicted) in enumerate(zip(groundTruth, predictions)):
        correct = "Yes" if actual == predicted else "No"
        print(f"{i:<6} {str(actual):<20} {str(predicted):<20} {correct}")

    print()

    #  compute counts
    total   = len(groundTruth)
    correct = sum(a == p for a, p in zip(groundTruth, predictions))
    wrong   = total - correct

    #  compute accuracy and error rate
    accuracy  = correct / total if total > 0 else 0.0
    errorRate = wrong   / total if total > 0 else 0.0

    # print summary stats
    print(f"Total records classified:{total}")
    print(f"Correctly classified:{correct}")
    print(f"Incorrectly classified:{wrong}")
    print(f"Accuracy:{accuracy:.4f}")
    print(f"Error rate:{errorRate:.4f}")

    #  build confusion matrix
    # rows = actual class, columns = predicted class
    classes = sorted(set(groundTruth) | set(predictions))
    matrix  = defaultdict(lambda: defaultdict(int))

    for actual, predicted in zip(groundTruth, predictions):
        matrix[actual][predicted] += 1

    #  print confusion matrix
    print("Confusion Matrix:")
    printConfusionMatrix(matrix, classes)


if __name__ == "__main__":
    main()

Writing predict.py


In [ ]:
%%writefile InduceC45.py
import sys
import json

from c45 import c45
from utils import parseCSV

def main():
  if len(sys.argv) < 2:
    print("Usage: python InduceC45.py <TrainingSet.csv> [outputTree.json]")
    sys.exit(1)

  csv = sys.argv[1]
  X, y, featureType, classVar = parseCSV(csv)

  if y is None:
    print("Error: training set must have class labels")
    sys.exit(1)

  model = c45(metric="InfoGain", threshold=0.0, featureType=featureType)
  tree = model.fit(X, y, data=csv)

  print(json.dumps(tree, indent=4))
  if len(sys.argv) > 2:
    output = sys.argv[2]
    model.save_tree(output)

if __name__ == "__main__":
  main()


Writing InduceC45.py


Test using small dataset like balloons.

In [ ]:
!python InduceC45.py adult+stretch.csv tree.json

{
    "dataset": "adult+stretch.csv",
    "node": {
        "var": "Act",
        "type": "categorical",
        "mostCommon": "F",
        "edges": [
            {
                "edge": {
                    "value": "STRETCH",
                    "node": {
                        "var": "Age",
                        "type": "categorical",
                        "mostCommon": "T",
                        "edges": [
                            {
                                "edge": {
                                    "value": "ADULT",
                                    "leaf": {
                                        "decision": "T",
                                        "p": 1.0
                                    }
                                }
                            },
                            {
                                "edge": {
                                    "value": "CHILD",
                                    "leaf": {
                       

In [ ]:
!python predict.py adult+stretch.csv tree.json

T
T
F
F
F
T
T
F
F
F
T
T
F
F
F
T
T
F
F
F


In [ ]:
!python predict.py adult+stretch.csv tree.json eval

Row    Actual               Predicted            Correct
------------------------------------------------------------
0      T                    T                    Yes
1      T                    T                    Yes
2      F                    F                    Yes
3      F                    F                    Yes
4      F                    F                    Yes
5      T                    T                    Yes
6      T                    T                    Yes
7      F                    F                    Yes
8      F                    F                    Yes
9      F                    F                    Yes
10     T                    T                    Yes
11     T                    T                    Yes
12     F                    F                    Yes
13     F                    F                    Yes
14     F                    F                    Yes
15     T                    T                    Yes
16     T                    T     

In [ ]:
%%writefile grid.json
{
  "InfoGain": [0.0, 0.01, 0.05, 0.1],
  "Ratio": [0.0, 0.01, 0.05, 0.1]
}

Writing grid.json


In [ ]:
%%writefile crossVal.py

import sys
import json
import random
from collections import defaultdict
from c45 import c45
from utils import parseCSV

def makeFolds(n, k=10, seed=42):
  """
  Split row indices into k folds
  """

  indices = list(range(n))

  random.seed(seed)
  random.shuffle(indices)

  foldList = [[] for num in range(k)]

  for i, idx in enumerate(indices):
    foldList[i % k].append(idx)

  return foldList

def buildConfusionMatrix(actual, predicted):
  """
  Build confusion matrix
  Rows - Columns
  actual - predicted
  """

  classes = sorted(set(actual) | set(predicted), key=str)
  matrix  = defaultdict(lambda: defaultdict(int))

  for act, pred in zip(actual, predicted):
    matrix[act][pred] += 1

  return matrix, classes

def printConfusionMatrix(matrix, classes):
  colW = max(len(str(c)) for c in classes) + 2
  header = "Actual \\ Predicted".ljust(colW)

  for c in classes:
    header += str(c).rjust(colW)
  print(header)
  print("-" * len(header))

  for act in classes:
    row = str(act).ljust(colW)

    for pred in classes:
      count = matrix[act][pred]
      row += str(count).rjust(colW)
    print(row)

def getAccuracy(act, pred):
  total = len(act)
  if total == 0:
    return 0.0

  correct = sum(a == p for a, p in zip(act, pred))
  return correct / total

def crossValidation(X, y, featureType, metric, threshold, folds):
  """
  Runs 10-fold cross validation
  """
  addPredictions = [None] * len(y)

  for f in folds:
    test = f
    testSet = set(f)

    train = []

    for i in range(len(y)):
      if i not in testSet:
        train.append(i)

    Xtrain = X.iloc[train].reset_index(drop=True)
    ytrain = y.iloc[train].reset_index(drop=True)
    Xtest = X.iloc[test].reset_index(drop=True)

    model = c45(metric=metric, threshold=threshold, featureType=featureType)
    model.fit(Xtrain, ytrain)

    preds = model.predict(Xtest)

    for i, pred in zip(test, preds):
      addPredictions[i] = pred

  return addPredictions

def main():
  if len(sys.argv) < 3:
    print("Usage: python crossVal.py <CSVFile> <GridFile> [outputTree.json]")
    sys.exit(1)

  csv = sys.argv[1]
  grid = sys.argv[2]

  X, y, featureType, classVar = parseCSV(csv)

  if y is None:
    print("Error: training set must have class labels")
    sys.exit(1)

  X = X.reset_index(drop=True)
  y = y.reset_index(drop=True)

  with open(grid, "r") as f:
    grid = json.load(f)

  foldList = makeFolds(len(y), k=10)

  bestMetric = None
  bestThreshold = None
  bestAccuracy = -1
  bestPred = None

  for m in ["InfoGain", "Ratio"]:
    if m not in grid:
      continue

    for t in grid[m]:
      print(f"{m}: {t}")
      preds = crossValidation(X, y, featureType, m, t, foldList)
      acc = getAccuracy(list(y), preds)

      print(f"Accuracy: {acc}")
      print()

      if acc > bestAccuracy:
        bestMetric = m
        bestThreshold = t
        bestAccuracy = acc
        bestPred = preds
  matrix, classes = buildConfusionMatrix(list(y), bestPred)
  print("Best Model")
  print("----------")
  print(f"Splitting Metric: {bestMetric}")
  print(f"Threshold: {bestThreshold}")
  print(f"Overall Cross-Validation Accuracy: {bestAccuracy:.4f}")
  print()
  print("Confusion Matrix:")
  printConfusionMatrix(matrix, classes)

if __name__ == "__main__":
  main()


Writing crossVal.py


More testing

In [ ]:
!python crossVal.py adult+stretch.csv grid.json

InfoGain: 0.0
Accuracy: 1.0

InfoGain: 0.01
Accuracy: 1.0

InfoGain: 0.05
Accuracy: 1.0

InfoGain: 0.1
Accuracy: 1.0

Ratio: 0.0
Accuracy: 1.0

Ratio: 0.01
Accuracy: 1.0

Ratio: 0.05
Accuracy: 1.0

Ratio: 0.1
Accuracy: 1.0

Best Model
----------
Splitting Metric: InfoGain
Threshold: 0.0
Overall Cross-Validation Accuracy: 1.0000

Confusion Matrix:
Actual \ Predicted  F  T
------------------------
F   12  0
T    0  8


In [ ]:
!python crossVal.py adult+stretch.csv grid.json bestTree.json

InfoGain: 0.0
Accuracy: 1.0

InfoGain: 0.01
Accuracy: 1.0

InfoGain: 0.05
Accuracy: 1.0

InfoGain: 0.1
Accuracy: 1.0

Ratio: 0.0
Accuracy: 1.0

Ratio: 0.01
Accuracy: 1.0

Ratio: 0.05
Accuracy: 1.0

Ratio: 0.1
Accuracy: 1.0

Best Model
----------
Splitting Metric: InfoGain
Threshold: 0.0
Overall Cross-Validation Accuracy: 1.0000

Confusion Matrix:
Actual \ Predicted  F  T
------------------------
F   12  0
T    0  8


Testing on main datasets

In [ ]:
!python crossVal.py iris.data.csv grid.json irisBestTree.json

InfoGain: 0.0
Accuracy: 0.9333333333333333

InfoGain: 0.01
Accuracy: 0.9333333333333333

InfoGain: 0.05
Accuracy: 0.9333333333333333

InfoGain: 0.1
Accuracy: 0.9333333333333333

Ratio: 0.0
Accuracy: 0.94

Ratio: 0.01
Accuracy: 0.94

Ratio: 0.05
Accuracy: 0.94

Ratio: 0.1
Accuracy: 0.94

Best Model
----------
Splitting Metric: Ratio
Threshold: 0.0
Overall Cross-Validation Accuracy: 0.9400

Confusion Matrix:
Actual \ Predicted      Iris-setosa  Iris-versicolor   Iris-virginica
---------------------------------------------------------------------
Iris-setosa                     50                0                0
Iris-versicolor                  0               45                5
Iris-virginica                   0                4               46


In [ ]:
!python crossVal.py nursery.csv grid.json nurseryBestTree.json

InfoGain: 0.0
Accuracy: 0.9888117283950617

InfoGain: 0.01
Accuracy: 0.9888117283950617

InfoGain: 0.05
Accuracy: 0.9875

InfoGain: 0.1
Accuracy: 0.983641975308642

Ratio: 0.0
Accuracy: 0.9883487654320988

Ratio: 0.01
Accuracy: 0.9871141975308642

Ratio: 0.05
Accuracy: 0.9835648148148148

Ratio: 0.1
Accuracy: 0.9436728395061729

Best Model
----------
Splitting Metric: InfoGain
Threshold: 0.0
Overall Cross-Validation Accuracy: 0.9888

Confusion Matrix:
Actual \ Predicted   not_recom    priority   recommend  spec_prior  very_recom
------------------------------------------------------------------------------
not_recom           4320           0           0           0           0
priority               0        4199           0          23          44
recommend              0           0           0           0           2
spec_prior             0          41           0        4003           0
very_recom             0          29           6           0         293


In [ ]:
!python crossVal.py letter-recognition.data.csv grid.json letterBestTree.json

InfoGain: 0.0
Accuracy: 0.88365

InfoGain: 0.01
Accuracy: 0.88365

InfoGain: 0.05
Accuracy: 0.8825

InfoGain: 0.1
Accuracy: 0.8795

Ratio: 0.0
Accuracy: 0.8662

Ratio: 0.01
Accuracy: 0.8662

Ratio: 0.05
Accuracy: 0.8661

Ratio: 0.1
Accuracy: 0.8657

Best Model
----------
Splitting Metric: InfoGain
Threshold: 0.0
Overall Cross-Validation Accuracy: 0.8837

Confusion Matrix:
Actual \ Predicted  A  B  C  D  E  F  G  H  I  J  K  L  M  N  O  P  Q  R  S  T  U  V  W  X  Y  Z
------------------------------------------------------------------------------------------------
A  742  0  2  3  1  0  3  1  2  3  2  3  2  5  3  0  2  1  4  0  3  0  0  4  3  0
B    0650  0 13  5  4  4  4  3  3  5  2  1  3  7  6  1 22  5  3  2 10  4  5  1  3
C    1  0644  0 18  6 21  5  2  1  4  4  0  2  5  0  4  2  1  7  2  1  4  1  0  1
D    2 19  0686  0  3  5 20  1  4  3  1  2 11 13  6  4 12  2  4  3  0  0  3  0  1
E    1  1  4  1674  1 10  1  0  0  8  7  1  0  0  4  7  5 14  7  0  2  0 14  1  5
F    3  7  1  2  1671

In [ ]:
%%writefile c45_results_notes.txt
C4.5 Results

Summary Table
Dataset                Best Metric     Best Threshold     10-Fold CV Accuracy
---------------------------------------------------------------------------
Iris                   Ratio           0.0                0.9400
Nursery                InfoGain        0.0                0.9888
Letter Recognition     InfoGain        0.0                0.8837


Dataset: Iris
Best Metric: Ratio
Best Threshold: 0.0
10-Fold Cross-Validation Accuracy: 0.9400

Confusion Matrix:
Actual \ Predicted      Iris-setosa  Iris-versicolor   Iris-virginica
---------------------------------------------------------------------
Iris-setosa                     50                0                0
Iris-versicolor                  0               45                5
Iris-virginica                   0                4               46

Notes:
The Iris model classified all Iris-setosa examples correctly. The only mistakes were between Iris-versicolor and Iris-virginica.


Dataset: Nursery
Best Metric: InfoGain
Best Threshold: 0.0
10-Fold Cross-Validation Accuracy: 0.9888

Confusion Matrix:
Actual \ Predicted   not_recom    priority   recommend  spec_prior  very_recom
------------------------------------------------------------------------------
not_recom           4320           0           0           0           0
priority               0        4199           0          23          44
recommend              0           0           0           0           2
spec_prior             0          41           0        4003           0
very_recom             0          29           6           0         293

Notes:
The Nursery model performed very well overall. The not_recom class was classified perfectly. The recommend class was not classified correctly, likely because it only had 2 examples.


Dataset: Letter Recognition
Best Metric: InfoGain
Best Threshold: 0.0
10-Fold Cross-Validation Accuracy: 0.8837

Notes:
The Letter Recognition dataset was harder than Iris and Nursery because it has 26 possible class labels and many more records. Information Gain performed better than Gain Ratio for this dataset. The confusion matrix is large, so the full version was saved separately in the terminal output file.

Writing c45_results_notes.txt


In [ ]:
%%writefile crossValSKL.py

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import sys
import json
import random
from collections import defaultdict
import pandas as pd
import matplotlib.pyplot as plt
from utils import parseCSV

def makeFolds(n, k=10, seed=42):
  """
  Split row indices into k folds.
  """

  indices = list(range(n))
  random.seed(seed)
  random.shuffle(indices)

  foldList = [[] for num in range(k)]

  for i, idx in enumerate(indices):
    foldList[i % k].append(idx)

  return foldList


def buildConfusionMatrix(actual, predicted):
  """
  Build confusion matrix.
  Rows - Columns
  Actual - Predicted
  """

  classes = sorted(set(actual) | set(predicted), key=str)
  matrix = defaultdict(lambda: defaultdict(int))

  for act, pred in zip(actual, predicted):
    matrix[act][pred] += 1

  return matrix, classes


def printConfusionMatrix(matrix, classes):
  colW = max(len(str(c)) for c in classes) + 2

  header = "Actual \\ Predicted".ljust(colW)

  for c in classes:
    header += str(c).rjust(colW)

  print(header)
  print("-" * len(header))

  for act in classes:
    row = str(act).ljust(colW)

    for pred in classes:
      count = matrix[act][pred]
      row += str(count).rjust(colW)
    print(row)


def getAccuracy(actual, predicted):
  total = len(actual)
  if total == 0:
    return 0.0

  return sum(a == p for a, p in zip(actual, predicted)) / total


def buildModel(featureType, threshold):
  """
  Categorical columns are converted to numbers and numeric columns are passed through
  """

  categoricalCols = []
  numericCols = []

  for col, kind in featureType.items():
    if kind == "categorical":
      categoricalCols.append(col)
    else:
      numericCols.append(col)

  preprocessor = ColumnTransformer(
    transformers=[
      (
        "cat",
        OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
        categoricalCols
      ),
      (
        "num",
        "passthrough",
        numericCols
      )
    ]
  )

  tree = DecisionTreeClassifier(
    criterion="entropy",
    min_impurity_decrease=threshold,
    random_state=42
  )

  model = Pipeline(
    steps=[
      ("preprocessor", preprocessor),
      ("tree", tree)
    ]
  )

  return model


def crossValidationSKL(X, y, featureType, threshold, foldList):
  """
  Run 10-fold cross-validation using Scikit-Learn
  """

  addPredictions = [None] * len(y)

  for fold in foldList:
    test = fold
    testSet = set(fold)

    train = []

    for i in range(len(y)):
      if i not in testSet:
        train.append(i)

    Xtrain = X.iloc[train].reset_index(drop=True)
    ytrain = y.iloc[train].reset_index(drop=True)
    Xtest = X.iloc[test].reset_index(drop=True)

    model = buildModel(featureType, threshold)

    model.fit(Xtrain, ytrain)

    preds = model.predict(Xtest)

    for i, pred in zip(test, preds):
      addPredictions[i] = pred

  return addPredictions


def saveFinalTreeImage(model, X, y, outputFile):
  """
  Saves the trained Scikit-Learn decision tree as an image.
  """

  treeModel = model.named_steps["tree"]

  plt.figure(figsize=(24, 14))

  plot_tree(
    treeModel,
    filled=True,
    rounded=True,
    class_names=[str(c) for c in sorted(set(y), key=str)]
  )

  plt.savefig(outputFile, bbox_inches="tight", dpi=200)
  plt.close()


def main():
  if len(sys.argv) < 3:
    print("Usage: python crossValSKL.py <CSVFile> <GridFile> [outputTree.png]")
    sys.exit(1)

  csv = sys.argv[1]
  gridFile = sys.argv[2]

  outputTreeFile = None

  if len(sys.argv) >= 4:
    outputTreeFile = sys.argv[3]

  X, y, featureType, classVar = parseCSV(csv)

  if y is None:
    print("Error: training set must have class labels")
    sys.exit(1)

  X = X.reset_index(drop=True)
  y = y.reset_index(drop=True)

  with open(gridFile, "r") as f:
    grid = json.load(f)

  foldList = makeFolds(len(y), k=10)
  bestThreshold = None
  bestAccuracy = -1
  bestPredictions = None

  # Scikit-Learn DecisionTreeClassifier supports entropy/InfoGain
  if "InfoGain" not in grid:
    print("Error: grid file must contain InfoGain list.")
    sys.exit(1)

  for threshold in grid["InfoGain"]:
    print(f"InfoGain: {threshold}")

    preds = crossValidationSKL(X, y, featureType, threshold, foldList)

    acc = getAccuracy(list(y), preds)

    print(f"Accuracy: {acc}")
    print()

    if acc > bestAccuracy:
      bestThreshold = threshold
      bestAccuracy = acc
      bestPredictions = preds

  matrix, classes = buildConfusionMatrix(list(y), bestPredictions)

  print("Best Scikit-Learn Model")
  print("-----------------------")
  print("Splitting Metric: InfoGain / entropy")
  print(f"Threshold: {bestThreshold}")
  print(f"Overall Cross-Validation Accuracy: {bestAccuracy:.4f}")
  print()
  print("Confusion Matrix:")
  printConfusionMatrix(matrix, classes)

  if outputTreeFile is not None:
    finalModel = buildModel(featureType, bestThreshold)
    finalModel.fit(X, y)

    saveFinalTreeImage(finalModel, X, y, outputTreeFile)

    print()
    print(f"Final Scikit-Learn tree image saved to {outputTreeFile}")

if __name__ == "__main__":
  main()

Writing crossValSKL.py


SKL Testing

In [ ]:
!python crossValSKL.py iris.data.csv grid.json irisSKLTree.png

InfoGain: 0.0
Accuracy: 0.9333333333333333

InfoGain: 0.01
Accuracy: 0.9333333333333333

InfoGain: 0.05
Accuracy: 0.9466666666666667

InfoGain: 0.1
Accuracy: 0.9333333333333333

Best Scikit-Learn Model
-----------------------
Splitting Metric: InfoGain / entropy
Threshold: 0.05
Overall Cross-Validation Accuracy: 0.9467

Confusion Matrix:
Actual \ Predicted      Iris-setosa  Iris-versicolor   Iris-virginica
---------------------------------------------------------------------
Iris-setosa                     50                0                0
Iris-versicolor                  0               44                6
Iris-virginica                   0                2               48

Final Scikit-Learn tree image saved to irisSKLTree.png


In [ ]:
!python crossValSKL.py nursery.csv grid.json nurserySKLTree.png

InfoGain: 0.0
Accuracy: 0.9970679012345679

InfoGain: 0.01
Accuracy: 0.8726851851851852

InfoGain: 0.05
Accuracy: 0.8587962962962963

InfoGain: 0.1
Accuracy: 0.6625

Best Scikit-Learn Model
-----------------------
Splitting Metric: InfoGain / entropy
Threshold: 0.0
Overall Cross-Validation Accuracy: 0.9971

Confusion Matrix:
Actual \ Predicted   not_recom    priority   recommend  spec_prior  very_recom
------------------------------------------------------------------------------
not_recom           4320           0           0           0           0
priority               0        4251           0           9           6
recommend              0           0           0           0           2
spec_prior             0          16           0        4028           0
very_recom             0           2           3           0         323

Final Scikit-Learn tree image saved to nurserySKLTree.png


In [ ]:
!python crossValSKL.py letter-recognition.data.csv grid.json letterSKLTree.png

InfoGain: 0.0
Accuracy: 0.8872

InfoGain: 0.01
Accuracy: 0.63965

InfoGain: 0.05
Accuracy: 0.3371

InfoGain: 0.1
Accuracy: 0.2361

Best Scikit-Learn Model
-----------------------
Splitting Metric: InfoGain / entropy
Threshold: 0.0
Overall Cross-Validation Accuracy: 0.8872

Confusion Matrix:
Actual \ Predicted  A  B  C  D  E  F  G  H  I  J  K  L  M  N  O  P  Q  R  S  T  U  V  W  X  Y  Z
------------------------------------------------------------------------------------------------
A  747  1  0  3  0  0  3  2  1  4  2  4  3  3  1  1  2  1  4  0  3  0  0  2  2  0
B    0645  0 18  7  4  5  6  1  5  4  2  2  2  5  6  1 22  4  2  1 11  5  4  1  3
C    2  1637  0 18  4 25  3  3  1  4  7  0  1  7  0  4  2  0  7  3  0  3  0  2  2
D    1 16  1680  0  3  6 23  1  4  4  4  3 15 10  7  2 12  4  3  4  0  0  2  0  0
E    0  1  6  1674  2  9  1  1  1  5  5  2  0  1  3  7  4 13  7  0  1  1 14  1  8
F    1 12  0  1  2676  3  1  1  6  2  0  3  4  1 25  0  1  9  6  0  5  1  1 12  2
G    2  8  9  6 13  46

In [ ]:
%%writefile skl_results_notes.txt
Scikit-Learn Results

Summary Table
Dataset                Best Metric / Criterion      Best Threshold     10-Fold CV Accuracy
------------------------------------------------------------------------------------------
Iris                   entropy (InfoGain)           0.05               0.9467
Nursery                entropy (InfoGain)           0.0                0.9971
Letter Recognition     entropy (InfoGain)           0.0                0.8872


Dataset: Iris
Best Metric / Criterion: entropy (InfoGain)
Best Threshold: 0.05
10-Fold Cross-Validation Accuracy: 0.9467

Confusion Matrix:
Actual \ Predicted      Iris-setosa  Iris-versicolor   Iris-virginica
---------------------------------------------------------------------
Iris-setosa                     50                0                0
Iris-versicolor                  0               44                6
Iris-virginica                   0                2               48

Notes:
The sklearn tree classified all Iris-setosa examples correctly. Most mistakes were again between Iris-versicolor and Iris-virginica.


Dataset: Nursery
Best Metric / Criterion: entropy (InfoGain)
Best Threshold: 0.0
10-Fold Cross-Validation Accuracy: 0.9971

Confusion Matrix:
Actual \ Predicted   not_recom    priority   recommend  spec_prior  very_recom
------------------------------------------------------------------------------
not_recom           4320           0           0           0           0
priority               0        4251           0           9           6
recommend              0           0           0           0           2
spec_prior             0          16           0        4028           0
very_recom             0           2           3           0         323

Notes:
The sklearn model performed extremely well on Nursery. The not_recom class was classified perfectly. The recommend class still remained difficult because it has very few examples.


Dataset: Letter Recognition
Best Metric / Criterion: entropy (InfoGain)
Best Threshold: 0.0
10-Fold Cross-Validation Accuracy: 0.8872

Notes:
The Letter Recognition dataset was the hardest of the three. Sklearn slightly outperformed the custom C4.5 implementation. The full confusion matrix is large and was saved separately in the terminal output file.

Writing skl_results_notes.txt


In [ ]:
%%writefile comparison_notes.txt
C4.5 vs Scikit-Learn Comparison

Summary Table
Dataset                Custom C4.5 Best Setting       Custom Accuracy     SKL Best Setting                SKL Accuracy
----------------------------------------------------------------------------------------------------------------------
Iris                   Ratio, threshold = 0.0          0.9400              entropy, threshold = 0.05       0.9467
Nursery                InfoGain, threshold = 0.0       0.9888              entropy, threshold = 0.0        0.9971
Letter Recognition     InfoGain, threshold = 0.0       0.8837              entropy, threshold = 0.0        0.8872

Writing comparison_notes.txt


In [ ]:
%%writefile README
CSC 466 Lab 2 - C4.5 Decision tree

Team Members:
- Name: Winnie Trinh
  Email: witrinh@calpoly.edu
- Name: Nathan Madlansacay
  Email: nmadlans@calpoly.edu

Files:
- c45.py
- utils.py
- InduceC45.py
- predict.py
- crossVal.py
- crossValSKL.py
- grid.json

To run:
- python InduceC45.py <TrainingSet.csv> [outputTree.json]
- python predict.py <CSVFile> <JSONFile> [eval]
- python crossVal.py <CSVFile> <GridFile> [outputTree.json]
- python crossValSKL.py <CSVFile> <GridFile> [outputTree.png]

Notes:
- The programs take the required input files followed by an optional output/evaluation argument
- InduceC45.py and crossVal.py can save a JSON tree when an optional output file is provided
- predict.py uses the optional argument "eval" to print evaluation statistics
- crossValSKL.py can save a Scikit-Learn tree image when an optional image filename is provided
- The code expects the 3-line CSV header format

Writing README


## Congratulations! You are done with Lab 2

In [ ]:
# @markdown Run this cell to download this notebook as a webpage, `_NOTEBOOK.html`.

import google, json, nbformat

# Get the current notebook and write it to _NOTEBOOK.ipynb
raw_notebook = google.colab._message.blocking_request("get_ipynb",
                                                      timeout_sec=30)["ipynb"]
with open("_NOTEBOOK.ipynb", "w", encoding="utf-8") as ipynb_file:
  ipynb_file.write(json.dumps(raw_notebook))

# Use nbconvert to convert .ipynb to .html.
!jupyter nbconvert --to html --log-level WARN _NOTEBOOK.ipynb

# Download the .html file.
google.colab.files.download("_NOTEBOOK.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

- Open `_NOTEBOOK.html` in your browser, and save it as a PDF.
    - Go to File > Print > Save as PDF.
- Double check that all of your code and output is visible in the saved PDF.
- Upload the PDF to Gradescope.
    - Please be sure to select the correct pages corresponding to each question.